# Wan 2.1 SteadyDancer — Перенос танца**Перенос движений из видео с танцем на любого персонажа.** Загрузите видео танца + фото человека → получите видео, где этот человек танцует.**Время настройки:** ~15-20 мин (скачивание моделей ~20 ГБ)**VRAM:** ~15 ГБ пиковое потребление (T4 16 ГБ — на пределе!)---### Как это работает```[1. GPU]  →  [2. Установка]  →  [3. Ноды]  →  [4. Модели]  →  [5. Воркфлоу]  →  [6. ЗАГРУЗКА]  →  [7. Запуск]                                                                                        ↑                                                                               Два файла нужны:                                                                          1. Видео с танцем (.mp4)                                                                          2. Фото персонажа (.jpg/.png)```---### Что делает этот ноутбук?```Видео с танцем  ──────────┐                          ├──→  [Детекция поз]  ──→  [SteadyDancer]  ──→  Видео с вашим персонажемФото персонажа  ──────────┘       (ViTPose)           (Wan 2.1)           танцующим тот же танец```| Этап | Описание ||------|----------|| **Детекция поз** | ViTPose + YOLOv10 извлекают скелет из видео с танцем || **Кодирование** | CLIP Vision + VAE кодируют фото персонажа || **Генерация** | SteadyDancer генерирует персонажа в позах из танца || **Выход** | H.264 MP4 видео 720x1280 @ 30fps со звуком из оригинала |---### Что загружать?| Файл | Формат | Требования ||------|--------|------------|| **Видео с танцем** | .mp4 | Полная фигура, стабильная камера, 5-15 сек || **Фото персонажа** | .jpg, .png | В полный рост или по пояс, чёткое, без водяных знаков |### Чего НЕ нужно- Видео с несколькими людьми (только 1 человек)- Фото с закрытыми частями тела (маски, громоздкая одежда)- Видео дольше 15 секунд (на T4 может не хватить памяти)

In [ ]:
#@title 1. Проверка GPU!nvidia-smi --query-gpu=name,memory.total --format=csv,noheaderimport torchprint(f"CUDA: {torch.cuda.is_available()}")if torch.cuda.is_available():    gpu = torch.cuda.get_device_name(0)    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3    print(f"GPU: {gpu}")    print(f"VRAM: {vram:.1f} ГБ")    if vram < 15:        print("\n⚠ SteadyDancer требует ~15 ГБ VRAM. Может быть тесно!")    else:        print("\nVRAM достаточно для SteadyDancer.")

In [ ]:
#@title 2. Установка ComfyUI%cd /content!git clone https://github.com/comfyanonymous/ComfyUI.git 2>/dev/null || echo "Уже склонировано"%cd /content/ComfyUI!pip install -r requirements.txt -qprint("\nComfyUI установлен!")

In [ ]:
#@title 3. Установка кастомных нод%cd /content/ComfyUI/custom_nodesrepos = {    "ComfyUI-WanVideoWrapper": "https://github.com/kijai/ComfyUI-WanVideoWrapper.git",    "ComfyUI-WanAnimatePreprocess": "https://github.com/kijai/ComfyUI-WanAnimatePreprocess.git",    "ComfyUI-KJNodes": "https://github.com/kijai/ComfyUI-KJNodes.git",    "ComfyUI-VideoHelperSuite": "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git",    "ComfyUI-Easy-Use": "https://github.com/yolain/ComfyUI-Easy-Use.git",}for name, url in repos.items():    !git clone {url} 2>/dev/null || echo "{name} уже склонирован"# Зависимостиimport osfor name in repos:    req = f"/content/ComfyUI/custom_nodes/{name}/requirements.txt"    if os.path.exists(req):        !pip install -r {req} -q# ONNX Runtime для детекции поз!pip install onnxruntime-gpu -qprint(f"\n{len(repos)} кастомных нод установлено!")

In [ ]:
#@title 4. Скачивание моделей (~20 ГБ, 10-15 мин)import osM = "/content/ComfyUI/models"os.makedirs(f"{M}/diffusion_models", exist_ok=True)os.makedirs(f"{M}/text_encoders", exist_ok=True)os.makedirs(f"{M}/vae", exist_ok=True)os.makedirs(f"{M}/clip_vision", exist_ok=True)os.makedirs(f"{M}/loras", exist_ok=True)os.makedirs(f"{M}/detection", exist_ok=True)KIJAI = "https://huggingface.co/Kijai/WanVideo_comfy/resolve/main"KIJAI_FP8 = "https://huggingface.co/Kijai/WanVideo_comfy_fp8_scaled/resolve/main"COMFY = "https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files"files = {    # SteadyDancer fp8 (~16 ГБ на диске, ~8 ГБ в VRAM с block swap)    f"{M}/diffusion_models/Wan21_SteadyDancer_fp8_e4m3fn_scaled_KJ.safetensors":        f"{KIJAI_FP8}/SteadyDancer/Wan21_SteadyDancer_fp8_e4m3fn_scaled_KJ.safetensors",    # Текстовый энкодер UMT5 XXL bf16 (~10 ГБ)    f"{M}/text_encoders/umt5-xxl-enc-bf16.safetensors":        f"{KIJAI}/umt5-xxl-enc-bf16.safetensors",    # VAE bf16 (~400 МБ)    f"{M}/vae/Wan2_1_VAE_bf16.safetensors":        f"{KIJAI}/Wan2_1_VAE_bf16.safetensors",    # CLIP Vision H (~1.3 ГБ)    f"{M}/clip_vision/clip_vision_h.safetensors":        f"{COMFY}/clip_vision/clip_vision_h.safetensors",    # LightX2V LoRA для 4-шаговой генерации (~738 МБ)    f"{M}/loras/lightx2v_I2V_14B_480p_cfg_step_distill_rank64_bf16.safetensors":        f"{KIJAI}/Lightx2v/lightx2v_I2V_14B_480p_cfg_step_distill_rank64_bf16.safetensors",    # ViTPose ONNX для детекции поз (~1.2 ГБ)    f"{M}/detection/vitpose-l-wholebody.onnx":        "https://huggingface.co/JunkyByte/easy_ViTPose/resolve/main/onnx/wholebody/vitpose-l-wholebody.onnx",    # YOLOv10m ONNX для детекции людей (~62 МБ)    f"{M}/detection/yolov10m.onnx":        "https://huggingface.co/Wan-AI/Wan2.2-Animate-14B/resolve/main/process_checkpoint/det/yolov10m.onnx",}for dest, url in files.items():    if os.path.exists(dest):        print(f"Уже существует: {os.path.basename(dest)}")    else:        print(f"\nСкачивание {os.path.basename(dest)}...")        !wget -q --show-progress -O "{dest}" "{url}"print("\nВсе модели готовы!")!du -sh {M}/diffusion_models/* {M}/text_encoders/* {M}/vae/* {M}/clip_vision/* {M}/loras/* {M}/detection/*

In [ ]:
#@title 5. Скачивание воркфлоуimport osGIST = "ea1ab4a53e1be1b8401e5031bcdae4f3"RAW = f"https://gist.githubusercontent.com/russiankendricklamar/{GIST}/raw"WF_DIR = "/content/ComfyUI/user/default/workflows"os.makedirs(WF_DIR, exist_ok=True)!wget -q -O "{WF_DIR}/video_wan_dancer.json" "{RAW}/video_wan_dancer.json"print("Скачано: video_wan_dancer.json")print(f"\nВоркфлоу сохранён в {WF_DIR}")

In [ ]:
#@title 6. Загрузка медиа (видео танца + фото персонажа)#@markdown ## Загрузите 2 файла:#@markdown 1. **Видео с танцем** (.mp4) — полная фигура, 1 человек, 5-15 сек#@markdown 2. **Фото персонажа** (.jpg/.png) — в полный рост или по поясfrom google.colab import filesfrom IPython.display import display, Image as IPImageimport osINPUT_DIR = "/content/ComfyUI/input"os.makedirs(INPUT_DIR, exist_ok=True)VALID_IMAGES = ('.jpg', '.jpeg', '.png')VALID_VIDEO = ('.mp4',)print("=" * 60)print("  ЗАГРУЗКА МЕДИА ДЛЯ STEADYDANCER")print("=" * 60)print(f"  Папка: {INPUT_DIR}")print()print("  Нужны 2 файла:")print("    1. Видео с танцем (.mp4)")print("    2. Фото персонажа (.jpg / .png)")print()uploaded = files.upload()images = []videos = []skipped = []for name, data in uploaded.items():    ext = os.path.splitext(name)[1].lower()    path = os.path.join(INPUT_DIR, name)    if ext in VALID_IMAGES:        with open(path, "wb") as f:            f.write(data)        images.append(name)    elif ext in VALID_VIDEO:        with open(path, "wb") as f:            f.write(data)        videos.append(name)    else:        skipped.append(name)print(f"\n{'=' * 60}")print("  РЕЗУЛЬТАТ ЗАГРУЗКИ")print(f"{'=' * 60}")if videos:    print(f"\n  Видео с танцем ({len(videos)}):")    for f in videos:        print(f"    {f}")if images:    print(f"\n  Фото персонажа ({len(images)}):")    for f in images:        print(f"    {f}")if skipped:    print(f"\n  Пропущено ({len(skipped)}):")    for f in skipped:        print(f"    {f}")# Готовностьif videos and images:    print("\n  Готово для SteadyDancer!")    print("  Выберите эти файлы в ComfyUI:")    print(f"    VHS_LoadVideo → {videos[0]}")    print(f"    LoadImage → {images[0]}")elif videos:    print("\n  Нужно ещё фото персонажа (.jpg/.png)")elif images:    print("\n  Нужно ещё видео с танцем (.mp4)")# Превью фотоif images:    print(f"\nПревью:")    for f in images[:2]:        path = os.path.join(INPUT_DIR, f)        try:            display(IPImage(filename=path, width=300))        except Exception:            pass

In [ ]:
#@title 7. Запуск ComfyUIimport subprocess, time, re# Cloudflared!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1# Запуск ComfyUIcomfy = subprocess.Popen(    ["python", "main.py", "--listen", "0.0.0.0", "--port", "8188"],    cwd="/content/ComfyUI",    stdout=subprocess.PIPE,    stderr=subprocess.STDOUT)print("Запуск ComfyUI... (30с)")time.sleep(30)# Туннельtunnel = subprocess.Popen(    ["cloudflared", "tunnel", "--url", "http://localhost:8188"],    stdout=subprocess.PIPE,    stderr=subprocess.STDOUT)time.sleep(5)url = Nonefor _ in range(20):    line = tunnel.stdout.readline().decode()    match = re.search(r"https://[\w-]+\.trycloudflare\.com", line)    if match:        url = match.group(0)        breakif url:    print(f"\n{'='*60}")    print(f"  ComfyUI готов!")    print(f"  Откройте: {url}")    print(f"{'='*60}")    print("\n  1. Меню → Load → video_wan_dancer.json")    print("  2. В ноде VHS_LoadVideo выберите видео с танцем")    print("  3. В ноде LoadImage выберите фото персонажа")    print("  4. Нажмите Queue Prompt")    print("\n  Генерация: ~5-10 мин на T4")    print("  Результат: MP4 видео 720x1280 @ 30fps")else:    print("Ссылка не найдена. ComfyUI на порту 8188.")

In [ ]:
#@title 8. Сохранение на Google Drivefrom google.colab import driveimport shutil, glob, osdrive.mount('/content/drive')src = "/content/ComfyUI/output"dst = "/content/drive/MyDrive/ComfyUI_Output/dancer"os.makedirs(dst, exist_ok=True)results = glob.glob(f"{src}/**/*.mp4", recursive=True)if not results:    print("Результатов пока нет. Сначала сгенерируйте видео!")else:    for f in results:        shutil.copy2(f, dst)        print(f"Скопировано: {os.path.basename(f)}")    print(f"\n{len(results)} файлов сохранено в {dst}")

---## Гайд по воркфлоу `video_wan_dancer`### Основные параметры| Параметр | Значение | Описание ||----------|----------|----------|| `num_frames` | 81 | Количество кадров в выходном видео || `steps` | 4 | Шаги генерации (LightX2V LoRA) || `CFG` | 1.0 | Для distill LoRA = 1.0 || `shift` | 5 | Сдвиг шедулера || `block_swap` | 20 | Оффлоад блоков для T4 || `context_length` | 81 | Размер контекстного окна || `fps` | 30 | Частота кадров выхода |### Советы для лучшего результата1. **Видео с танцем:**   - Один человек в кадре, полная фигура   - Стабильная камера (без тряски)   - Хорошее освещение, контрастный фон   - Длина 5-10 секунд (больше = дольше + больше VRAM)2. **Фото персонажа:**   - Полный рост или хотя бы по пояс   - Нейтральная поза (стоя)   - Чёткое фото, хорошее освещение   - Без сложного фона### Если получается OOM1. Уменьшите `num_frames` до 49 (меньше кадров)2. Увеличьте `block_swap` до 303. Используйте более короткое видео-источник4. Перезапустите runtime: Runtime → Restart runtime